# Grafos interactivos y matriz MGCECDL de variables CHEC

Notebook autosuficiente para generar tres grafos desde `site/data/variables.json`:

1. El grafo completo con 156 nodos.
2. El grafo filtrado con las filas que tengan `SELECCIÓN = 1` en el Excel vigente.
3. El grafo MGCECDL con las variables que sobreviven a `procesar_dataset_completo`, excluyendo `UITI_VANO` por ser el objetivo.

El notebook prioriza el archivo `data/Variables_seleccion.xlsx` del proyecto `chec_impacto`. Las conexiones entre variables seleccionadas se preservan atravesando variables retiradas y usando como peso efectivo el mínimo peso del camino original.

También regenera la matriz dirigida de adyacencia en el orden exacto de `features` y guarda:

- `data/graphs/mgcecdl_adjacency_matrix.npy`
- `data/graphs/mgcecdl_feature_order.json`
- `data/graphs/mgcecdl_preserved_edges.json`

In [1]:
# Ejecutar solo si el entorno no tiene instaladas estas dependencias.
%pip install networkx pyvis openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import sys
from pathlib import Path

import numpy as np

import networkx as nx
from openpyxl import load_workbook
from pyvis.network import Network


# Sube desde el cwd hasta encontrar site/data/variables.json, para que el notebook
# funcione sin importar desde qué directorio se ejecute (Jupyter local o Colab/Kaggle).
def find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'site/data/variables.json').is_file():
            return candidate
    raise FileNotFoundError(
        'No se encontró site/data/variables.json en el directorio actual ni en sus padres.'
    )


REPO_ROOT = find_repo_root()
JSON_PATH = REPO_ROOT / 'site/data/variables.json'
# Salidas HTML interactivas (pyvis) para inspección visual de los 3 grafos.
FULL_OUTPUT = REPO_ROOT / 'src/assets/site/results/grafo_red_nivel2.html'
SELECTED_OUTPUT = REPO_ROOT / 'src/assets/site/results/grafo_red_nivel2_seleccionadas.html'
MGCECDL_OUTPUT = REPO_ROOT / 'src/assets/site/results/grafo_red_nivel2_mgcecdl.html'
MGCECDL_PROJECT_ROOT = REPO_ROOT

# Color por modo (A-F) del diccionario de variables, usado en los grafos y su leyenda.
MODE_COLORS = {
    'A': '#e74c3c',
    'B': '#f39c12',
    'C': '#9b59b6',
    'D': '#3498db',
    'E': '#1abc9c',
    'F': '#2ecc71',
}

In [3]:

# El path puede venir de una variable de entorno (útil en CI/Colab) o de las
# ubicaciones habituales del proyecto chec_impacto.
def resolve_selection_path():
    candidates = [
        os.environ.get('VARIABLES_SELECCION_PATH'),
        MGCECDL_PROJECT_ROOT / 'data/Variables_seleccion.xlsx',
        REPO_ROOT / 'data/Variables_seleccion.xlsx',
    ]
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate)
    raise FileNotFoundError(
        'No se encontró Variables_seleccion.xlsx. Define VARIABLES_SELECCION_PATH '
        'o ubica el archivo en data/Variables_seleccion.xlsx.'
    )


# Lee la hoja 'Variables_análisis' del Excel de expertos y devuelve el conjunto
# de nombres de columna marcados con SELECCIÓN=1 (variable vigente para el modelo).
def load_selected_names(path):
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        worksheet = workbook['Variables_análisis']
        headers = {str(cell.value).strip(): index for index, cell in enumerate(worksheet[1])}
        variable_index = headers['COLUMNA']
        selection_index = headers['SELECCIÓN']

        selected = set()
        for row in worksheet.iter_rows(min_row=2, values_only=True):
            variable = row[variable_index]
            selection = row[selection_index]
            if variable and str(selection).strip().lower() in {'1', '1.0', 'true', 'sí', 'si'}:
                selected.add(str(variable).strip())
        return selected
    finally:
        workbook.close()


# Expande la configuración de variables.json (compacta por modo) a la lista real
# de columnas por modo. El modo F es especial: además de sus variables estáticas,
# genera una columna por hora de ventana climática (family_0 ... family_{ventana-1}).
def expand_variables(config):
    variables_by_mode = {}
    climate_families = set()
    window_hours = config['ventanaClimaticaHoras']

    for mode in config['modos']:
        mode_id = mode['id']
        if mode_id == 'F':
            climate_families.update(mode['familiasClimaticas'])
            variables = list(mode['variablesEstaticas'])
            for family in mode['familiasClimaticas']:
                variables.extend(f'{family}_{hour}' for hour in range(window_hours))
        else:
            variables = list(mode['variables'])

        # Verificación de consistencia contra el conteo declarado en variables.json.
        assert len(variables) == mode['cantidad'], (
            f"El modo {mode_id} declara {mode['cantidad']} variables, "
            f"pero se generaron {len(variables)}."
        )
        variables_by_mode[mode_id] = variables

    return variables_by_mode, climate_families


# Filtra cada modo a solo las variables presentes en el Excel de selección.
# Las columnas climáticas por hora (family_N) se incluyen si su familia base
# (sin sufijo de hora) está seleccionada.
def selected_variables(variables_by_mode, selected_names, climate_families):
    return {
        mode_id: [
            variable
            for variable in variables
            if variable in selected_names
            or any(
                variable.startswith(f'{family}_') and family in selected_names
                for family in climate_families
            )
        ]
        for mode_id, variables in variables_by_mode.items()
    }

In [4]:
# Aristas dirigidas (source, target, peso) del grafo experto de 156 variables.
# El peso no es una probabilidad estadística: es una fuerza de relación asignada
# a mano por el equipo de dominio, y es lo que luego regulariza el entrenamiento
# de MGCECDL (graph regularizer) vía la matriz de adyacencia derivada de esta lista.
def build_edges(config):
    edges = []
    window_hours = config['ventanaClimaticaHoras']
    climate_families = next(
        mode['familiasClimaticas'] for mode in config['modos'] if mode['id'] == 'F'
    )

    # Cadena temporal por familia climática: cada hora se conecta con la hora
    # previa (family_N -> family_{N-1} ... -> family_0), y family_0 conecta con
    # COD_CAUSA como punto de entrada al resto del grafo causal.
    for family in climate_families:
        for hour in range(window_hours - 1, 0, -1):
            edges.append((f'{family}_{hour}', f'{family}_{hour - 1}', 0.90))
        edges.append((f'{family}_0', 'COD_CAUSA', 0.85))

    # Resto de aristas del grafo experto (variables estructurales, de red y de
    # usuarios) hasta converger en UITI_VANO, la variable objetivo del modelo.
    edges.extend([
        ('NR_T', 'COD_CAUSA', 0.85), ('DDT', 'COD_CAUSA', 0.90),
        ('wind_gust_spd_0', 'NR_T', 0.80), ('CANTIDAD_TIERRA', 'DDT', 0.85),
        ('NG_RED', 'DDT', 0.75), ('LONGITUD', 'COD_CAUSA', 0.70),
        ('CONDUCTOR', 'COD_CAUSA', 0.80), ('ALTURA', 'NR_T', 0.75),
        ('VAL_CRIT_APOYO', 'NR_T', 0.60), ('CLASE', 'VAL_CRIT_APOYO', 0.80),
        ('NORMA', 'VAL_CRIT_APOYO', 0.80), ('X1', 'FID_VANO', 1.0),
        ('Y1', 'FID_VANO', 1.0), ('X2', 'FID_VANO', 1.0),
        ('Y2', 'FID_VANO', 1.0), ('FID_VANO', 'LVSW', 0.9),
        ('CIRCUITO', 'FID_VANO', 0.8), ('LVSW', 'COD_EQ_PROTEGE', 0.85),
        ('CNT_VN', 'COD_EQ_PROTEGE', 0.85), ('COD_EQ_PROTEGE', 'FID_SW', 1.0),
        ('FID_SW', 'TIPO', 0.9), ('TIPO', 'DURACION', 0.85),
        ('TIPO', 'T_USUS_EQ_PROT', 0.85), ('CNT_VN_SW', 'T_USUS_EQ_PROT', 0.80),
        ('PORC_APORTE_VANO', 'CNT_VN_SW', 0.7),
        ('COD_APOYO_FIN', 'FID_APOYO_FIN', 1.0),
        ('PROPIETARIO', 'FID_APOYO_FIN', 0.5),
        ('ELEMENTO', 'FID_APOYO_FIN', 0.8),
        ('LONG_CRUCETA', 'FID_APOYO_FIN', 0.7),
        ('FID_TRAFO', 'CODIGO', 1.0), ('FID_APOYO_FIN', 'FID_TRAFO', 0.9),
        ('CAPACIDAD_NOMINAL', 'CNT_USUS', 0.85),
        ('FECHA_OPERACION_TRF', 'FID_TRAFO', 0.6),
        ('PROMEDIO_KWH_TRF', 'CNT_USUS', 0.8), ('CNT_USUS', 'TOT_USUS', 0.9),
        ('CNT_TRF', 'TOT_USUS', 0.85), ('PROMEDIO_KWH_VANO', 'CNT_FASES', 0.7),
        ('CALIBRE_NEUTRO', 'CONDUCTOR', 0.8),
        ('FECHA_OPERACION_VANO', 'CONDUCTOR', 0.6),
        ('TIPO_TAX', 'FID_VANO', 0.7), ('COD_CAUSA', 'DESC_CAUSA', 1.0),
        ('COD_CAUSA', 'FECHA', 0.7), ('FECHA', 'UITI_VANO', 0.7),
        ('T_USUS_EQ_PROT', 'TOT_USUS', 0.95),
        ('DURACION', 'UITI', 1.0), ('TOT_USUS', 'UITI', 1.0),
        ('UITI', 'UITI_VANO', 1.0),
        ('PORC_APORTE_VANO', 'UITI_VANO', 1.0),
    ])
    return edges

In [5]:
from collections import deque


def _format_weight(weight):
    return f'{weight:.2f}' if isinstance(weight, (int, float)) else str(weight)


def _register_preserved_edge(preserved_edges, source, target, weight, path, is_virtual):
    """
    Registra una arista directa o preservada.

    Si existe más de un camino entre dos nodos seleccionados, se conserva:
    1. La arista directa, si existe.
    2. En caso contrario, el camino preservado con mayor peso efectivo.

    El peso efectivo de un camino filtrado se calcula como el mínimo peso
    de sus aristas originales, es decir, la conexión más débil del camino.
    """
    if source == target:
        return

    key = (source, target)
    current = preserved_edges.get(key)

    candidate = {
        'source': source,
        'target': target,
        'weight': weight,
        'path': path,
        'is_virtual': is_virtual,
    }

    if current is None:
        preserved_edges[key] = candidate
        return

    # Una arista directa tiene prioridad sobre una arista reconstruida.
    if current['is_virtual'] and not is_virtual:
        preserved_edges[key] = candidate
        return

    # Si ambas son del mismo tipo, se conserva la de mayor peso efectivo.
    if current['is_virtual'] == is_virtual and weight > current['weight']:
        preserved_edges[key] = candidate


def build_preserved_edges(edges, kept_nodes):
    """
    Construye las aristas del grafo filtrado preservando conectividad.

    Regla:
    - Si source y target están en kept_nodes, se conserva la arista original.
    - Si una ruta source -> ... -> target pasa únicamente por nodos filtrados
      como intermediarios, se agrega una arista directa source -> target.

    Ejemplo:
    a -> b -> c -> d, con kept_nodes = {a, d}, produce a -> d.
    """
    kept_nodes = set(kept_nodes)

    adjacency = {}
    for source, target, weight in edges:
        adjacency.setdefault(source, []).append((target, weight))

    preserved_edges = {}

    # BFS desde cada nodo conservado: mientras se atraviesan nodos retirados
    # se va acumulando el peso mínimo del camino (cuello de botella), y el
    # recorrido se corta apenas se alcanza otro nodo conservado.
    for source in kept_nodes:
        queue = deque()
        visited_removed = set()

        for target, weight in adjacency.get(source, []):
            if target in kept_nodes:
                _register_preserved_edge(
                    preserved_edges,
                    source,
                    target,
                    weight,
                    [source, target],
                    is_virtual=False,
                )
            else:
                queue.append((target, [source, target], weight))

        while queue:
            current_node, path, path_weight = queue.popleft()

            # Los nodos seleccionados solo pueden ser extremos del camino.
            # No se atraviesan para crear atajos más largos.
            if current_node in kept_nodes:
                _register_preserved_edge(
                    preserved_edges,
                    source,
                    current_node,
                    path_weight,
                    path,
                    is_virtual=True,
                )
                continue

            if current_node in visited_removed:
                continue
            visited_removed.add(current_node)

            for next_node, edge_weight in adjacency.get(current_node, []):
                effective_weight = min(path_weight, edge_weight)
                queue.append((next_node, [*path, next_node], effective_weight))

    return list(preserved_edges.values())


# Construye el grafo networkx para un subconjunto de variables. Cuando
# preserve_filtered_connections=True usa build_preserved_edges para no perder
# conectividad al filtrar nodos (aristas "virtuales" en línea punteada);
# cuando es False solo conserva aristas cuyos dos extremos siguen presentes.
def create_graph(
    variables_by_mode,
    edges,
    mode_names,
    preserve_filtered_connections=False,
):
    graph = nx.DiGraph()

    for mode_id, variables in variables_by_mode.items():
        for variable in variables:
            graph.add_node(
                variable,
                color=MODE_COLORS[mode_id],
                title=f'{variable}\n({mode_names[mode_id]})',
                size=15,
            )

    if preserve_filtered_connections:
        graph_edges = build_preserved_edges(edges, graph.nodes)
        for edge in graph_edges:
            source = edge['source']
            target = edge['target']
            weight = edge['weight']

            if edge['is_virtual']:
                path_text = ' → '.join(edge['path'])
                graph.add_edge(
                    source,
                    target,
                    weight=weight,
                    title=(
                        'Conexión preservada después del filtro\n'
                        f'Camino original: {path_text}\n'
                        f'Peso efectivo: {_format_weight(weight)}'
                    ),
                    dashes=True,
                )
            else:
                graph.add_edge(
                    source,
                    target,
                    weight=weight,
                    title=(
                        f'Conexión directa\n'
                        f'{source} → {target}\n'
                        f'Peso: {_format_weight(weight)}'
                    ),
                )
    else:
        for source, target, weight in edges:
            if source in graph and target in graph:
                graph.add_edge(
                    source,
                    target,
                    weight=weight,
                    title=(
                        f'Conexión directa\n'
                        f'{source} → {target}\n'
                        f'Peso: {_format_weight(weight)}'
                    ),
                )

    return graph


# Renderiza el grafo a HTML interactivo (pyvis) con física barnes_hut y le
# inyecta una leyenda HTML manual (conteo de nodos por modo + aristas
# preservadas) directamente en el <body> generado por pyvis.
def save_graph(graph, output_path, variables_by_mode, mode_names, title):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    network = Network(
        height='900px',
        width='100%',
        bgcolor='#ffffff',
        font_color='black',
        directed=True,
        cdn_resources='remote',
    )
    network.barnes_hut(
        gravity=-2000,
        central_gravity=0.1,
        spring_length=150,
        spring_strength=0.05,
        damping=0.9,
    )
    network.from_nx(graph)
    network.save_graph(str(output_path))

    legend = [
        '<div style="position:absolute;top:20px;left:20px;z-index:1000;'
        'background:rgba(255,255,255,.95);padding:15px;border-radius:8px;'
        'box-shadow:0 4px 15px rgba(0,0,0,.15);font-family:Segoe UI,Arial,sans-serif;'
        'font-size:14px;border:1px solid #e0e0e0;">',
        f'<h3 style="margin:0 0 12px;font-size:16px;color:#2c3e50;border-bottom:'
        f'2px solid #eee;padding-bottom:5px;">{title}</h3>',
    ]
    for mode_id, color in MODE_COLORS.items():
        count = len(variables_by_mode[mode_id])
        if count == 0:
            continue
        legend.append(
            f'<div style="display:flex;align-items:center;margin-bottom:6px;">'
            f'<span style="display:inline-block;width:14px;height:14px;background:{color};'
            'border-radius:50%;margin-right:10px;border:1px solid rgba(0,0,0,.2);"></span>'
            f'Modo {mode_id}: {mode_names[mode_id]} ({count})</div>'
        )

    virtual_edges = sum(
        1
        for _, _, attributes in graph.edges(data=True)
        if attributes.get('dashes')
    )
    if virtual_edges:
        legend.append(
            '<div style="margin-top:10px;padding-top:8px;border-top:1px solid #eee;">'
            '<span style="display:inline-block;width:22px;border-top:2px dashed #555;'
            'margin-right:8px;vertical-align:middle;"></span>'
            f'Conexiones preservadas por filtro ({virtual_edges})</div>'
        )

    legend.append('</div>')

    # pyvis ya escribió el HTML a disco; se reabre para insertar la leyenda y
    # quitar la referencia a un script local (lib/bindings/utils.js) que no
    # existe cuando el HTML se sirve de forma standalone.
    html = output_path.read_text(encoding='utf-8')
    html = html.replace('<body>', f"<body>\n{''.join(legend)}", 1)
    html = html.replace('<script src="lib/bindings/utils.js"></script>', '')
    html = '\n'.join(line.rstrip() for line in html.splitlines()) + '\n'
    output_path.write_text(html, encoding='utf-8')

In [6]:
with JSON_PATH.open(encoding='utf-8') as file:
    config = json.load(file)

selection_path = resolve_selection_path()
selected_names = load_selected_names(selection_path)
variables_by_mode, climate_families = expand_variables(config)
filtered_by_mode = selected_variables(
    variables_by_mode, selected_names, climate_families
)
mode_names = {mode['id']: mode['nombre'] for mode in config['modos']}
edges = build_edges(config)

# Grafo 1: las 156 variables completas del diccionario, sin filtrar.
full_graph = create_graph(variables_by_mode, edges, mode_names)
# Grafo 2: solo las variables marcadas SELECCIÓN=1 en el Excel, preservando
# conectividad entre ellas a través de las variables retiradas (ver build_preserved_edges).
filtered_graph = create_graph(
    filtered_by_mode,
    edges,
    mode_names,
    preserve_filtered_connections=True,
)

# Chequeos de coherencia contra los conteos declarados en variables.json/Excel.
assert full_graph.number_of_nodes() == config['totalVariables']
expected_selected_nodes = sum(len(variables) for variables in filtered_by_mode.values())
assert filtered_graph.number_of_nodes() == expected_selected_nodes

save_graph(
    full_graph,
    FULL_OUTPUT,
    variables_by_mode,
    mode_names,
    'Diccionario de modos - Grafo completo',
)
save_graph(
    filtered_graph,
    SELECTED_OUTPUT,
    filtered_by_mode,
    mode_names,
    'Diccionario de modos - Variables seleccionadas',
)

print(f'Excel de selección: {selection_path}')
print(f'Variables marcadas en el Excel: {len(selected_names)}')
print(
    f'Grafo completo: {full_graph.number_of_nodes()} nodos, '
    f'{full_graph.number_of_edges()} conexiones -> {FULL_OUTPUT}'
)
print(
    f'Grafo seleccionado: {filtered_graph.number_of_nodes()} nodos, '
    f'{filtered_graph.number_of_edges()} conexiones '
    f'({sum(1 for _, _, data in filtered_graph.edges(data=True) if data.get("dashes"))} preservadas por filtro) '
    f'-> {SELECTED_OUTPUT}'
)

# Grafo y matriz usados por MGCECDL, alineados con el preprocesamiento real.
src_path = MGCECDL_PROJECT_ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
for module_name in list(sys.modules):
    if module_name == 'chec_impacto' or module_name.startswith('chec_impacto.'):
        del sys.modules[module_name]

from chec_impacto.data import (
    construir_matriz_adyacencia_mgcecdl,
    procesar_dataset_completo,
)

# Ejecuta el mismo preprocesamiento real que usa el entrenamiento (03_mgcecdl_training)
# para obtener el orden exacto de columnas ('features') que MGCECDL consume, ya que
# la matriz de adyacencia debe alinearse fila a fila con ese orden.
datos_procesados = procesar_dataset_completo(
    path_clima=MGCECDL_PROJECT_ROOT / 'data/Indicadores_vano_v3.csv',
    path_variables_seleccion=selection_path,
    use_sampling=False,
    min_samples_per_codigo=5,
    target='UITI_VANO',
    filtro_uiti_max=None,
    ventana_climatica_horas=config['ventanaClimaticaHoras'],
)
features = datos_procesados['features']
feature_set = set(features)
mgcecdl_by_mode = {
    mode_id: [variable for variable in variables if variable in feature_set]
    for mode_id, variables in variables_by_mode.items()
}
# Grafo 3: solo las variables que sobreviven al preprocesamiento real de MGCECDL
# (excluye UITI_VANO por ser el objetivo, no un feature de entrada).
mgcecdl_graph = create_graph(
    mgcecdl_by_mode,
    edges,
    mode_names,
    preserve_filtered_connections=True,
)
save_graph(
    mgcecdl_graph,
    MGCECDL_OUTPUT,
    mgcecdl_by_mode,
    mode_names,
    'Grafo MGCECDL - Orden del preprocesamiento',
)

# construir_matriz_adyacencia_mgcecdl (misma función de chec_impacto.data usada
# por el notebook de entrenamiento) recalcula las aristas preservadas y las
# vuelca en una matriz cuadrada en el orden exacto de 'features'. Estos 3
# artefactos son el producto real que consume el graph regularizer de MGCECDL;
# todo lo anterior en este notebook es preparación/validación visual.
adjacency_matrix, preserved_edges = construir_matriz_adyacencia_mgcecdl(
    features,
    ventana_climatica_horas=config['ventanaClimaticaHoras'],
)
graph_dir = MGCECDL_PROJECT_ROOT / 'data/graphs'
graph_dir.mkdir(parents=True, exist_ok=True)
np.save(graph_dir / 'mgcecdl_adjacency_matrix.npy', adjacency_matrix)
(graph_dir / 'mgcecdl_feature_order.json').write_text(
    json.dumps(features, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
(graph_dir / 'mgcecdl_preserved_edges.json').write_text(
    json.dumps(preserved_edges, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

assert adjacency_matrix.shape == (len(features), len(features))
assert set(mgcecdl_graph.nodes) == feature_set
assert 'UITI_VANO' not in feature_set
print(
    f'Grafo MGCECDL: {mgcecdl_graph.number_of_nodes()} nodos, '
    f'{mgcecdl_graph.number_of_edges()} conexiones -> {MGCECDL_OUTPUT}'
)
print(
    f'Matriz MGCECDL: {adjacency_matrix.shape}, '
    f'{np.count_nonzero(adjacency_matrix)} aristas -> {graph_dir}'
)

Excel de selección: /Users/diego/Desktop/Proyectos/chec-local-uiti-vano-interpreter/data/Variables_seleccion.xlsx
Variables marcadas en el Excel: 27
Grafo completo: 156 nodos, 156 conexiones -> /Users/diego/Desktop/Proyectos/chec-local-uiti-vano-interpreter/src/assets/site/results/grafo_red_nivel2.html
Grafo seleccionado: 71 nodos, 68 conexiones (16 preservadas por filtro) -> /Users/diego/Desktop/Proyectos/chec-local-uiti-vano-interpreter/src/assets/site/results/grafo_red_nivel2_seleccionadas.html
Cargando datos...
Procesamiento completado.
Shape X: (159470, 70), Shape y: (159470, 1)
Grafo MGCECDL: 70 nodos, 56 conexiones -> /Users/diego/Desktop/Proyectos/chec-local-uiti-vano-interpreter/src/assets/site/results/grafo_red_nivel2_mgcecdl.html
Matriz MGCECDL: (70, 70), 56 aristas -> /Users/diego/Desktop/Proyectos/chec-local-uiti-vano-interpreter/data/graphs
